## <a name="0">ステップ 0: ラボノートブックへようこそ</a>

task1 の回答を取得するために、セル内で次のコードを実行してください

In [ ]:
import base64
from config import ENV

decoded = base64.b64decode(ENV.get("TASK_1_ANSWER"))
s_decoded = decoded.decode()
print('task1 answer:', s_decoded)

task1 の回答が得られたら、回答フォームにコピー＆ペーストしてください！
こんなの朝飯前ですね！

## <a name="0">Amazon SageMaker での LLM のファインチューニング</a>

このソリューションでは、事前学習済みの大規模言語モデル（LLM）をファインチューニングする方法を探ります。これは人工知能（AI）の分野で強力な技術です。LLM は膨大な量のデータで事前学習されており、言語のニュアンスを理解し、一貫性のある応答を生成することに非常に効果的です。これらのモデルはデータから有用な特徴やパターンを抽出することを学習しており、様々な機械学習（ML）タスクにとって貴重なリソースとなっています。

ファインチューニング（転移学習とも呼ばれる）により、事前学習済みモデルで得られた知識を異なるが関連するタスクに適用することができます。モデルを一から学習させるのではなく、事前学習済みモデルから始めて、特定の問題領域に適応するように修正します。このアプローチは計算リソースを大幅に節約し、事前学習済みモデルの一般化能力の恩恵を受けることができます。

このノートブックでは、事前学習済みモデルをファインチューニングするステップバイステップのプロセスを説明します。以下の主要なステップを取り上げています：

1. <a href="#step1">ライブラリのインポート</a>
2. <a href="#step2">トレーニングデータセットの準備</a>
3. <a href="#step3">事前学習済み LLM の読み込み</a>
4. <a href="#step4">トレーナーの定義と LLM のファインチューニング</a>
5. <a href="#step5">ファインチューニングされたモデルのデプロイ</a>
6. <a href="#step6">デプロイされた推論のテスト</a>


注意：このノートブックは上から下へと進め、セクションをスキップしないでください。そうしないとコードの欠落によりエラーメッセージが表示される可能性があります。


## <a name="step1">ステップ 1: ライブラリのインポート</a>

以下の2つのコードブロックを順番に1つずつ実行して、Hugging Face Transformers ライブラリと、Transformers の依存関係である PyTorch ライブラリを含む必要なライブラリをインポートします。


In [ ]:
%%capture
pip install -r requirements.txt

In [ ]:
%%capture

import os
import numpy as np
import pandas as pd
from typing import Any, Dict, List, Tuple, Union
from datasets import Dataset, load_dataset, disable_caching
disable_caching() ## disable huggingface cache

from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer
from transformers import TextDataset

import torch
from torch.utils.data import Dataset, random_split
from transformers import TrainingArguments, Trainer
import accelerate
import bitsandbytes

from IPython.display import Markdown

import warnings
warnings.filterwarnings('ignore')

## <a name="step2">ステップ 2: トレーニングデータセットの準備</a>

データセットを読み込んで表示します。この実習では、メインデータセットとして [Amazon SageMaker FAQs](https://aws.amazon.com/sagemaker/faqs/) を使用します。このデータセットには `instruction` と `response` の2つの列があります。

In [ ]:
sagemaker_faqs_dataset = load_dataset("csv",
                                      data_files='data/amazon_sagemaker_faqs.csv')['train']
sagemaker_faqs_dataset

In [ ]:
sagemaker_faqs_dataset[0]

### <a name="step2">ステップ 2.1: プロンプトの準備</a>
LLM をファインチューニングするには、以下に示すように、指示データセットを PROMPT で装飾する必要があります。

In [ ]:
from utils.helpers import INTRO_BLURB, INSTRUCTION_KEY, RESPONSE_KEY, END_KEY, RESPONSE_KEY_NL, DEFAULT_SEED, PROMPT
'''
PROMPT = """{intro}
            {instruction_key}
            {instruction}
            {response_key}
            {response}
            {end_key}"""
'''
Markdown(PROMPT)

次に、`_add_text` 関数を通じて PROMPT をデータセットに供給します。この関数はレコードを入力として受け取ります。関数は両方のフィールド（instruction/response）が null でないことを確認し、値を事前定義された PROMPT テンプレートに渡します。

In [ ]:
def _add_text(rec):
    instruction = rec["instruction"]
    response = rec["response"]

    if not instruction:
        raise ValueError(f"Expected an instruction in: {rec}")

    if not response:
        raise ValueError(f"Expected a response in: {rec}")

    rec["text"] = PROMPT.format(
        instruction=instruction, response=response)

    return rec

In [ ]:
sagemaker_faqs_dataset = sagemaker_faqs_dataset.map(_add_text)
sagemaker_faqs_dataset[0]

テキストを綺麗に表示するために `Markdown` を使用します。

In [ ]:
Markdown(sagemaker_faqs_dataset[0]['text'])

## <a name="#step3">ステップ 3: 事前学習済み LLM の読み込み</a>


事前学習済みモデルを読み込むには、Hugging Face Transformers ライブラリの `databricks/dolly-v2-3b` モデルを使用して、トークナイザーとベースモデルを初期化します。トークナイザーは生のテキストをトークンに変換し、ベースモデルは与えられたプロンプトに基づいてテキストを生成します。前述の手順に従うことで、これらのコンポーネントを正しくインスタンス化し、コード内でその機能を使用することができます。


`AutoTokenizer.from_pretrained()` 関数はトークナイザーをインスタンス化するために使用されます。
- `padding_side="left"` はパディングトークンが追加されるシーケンスの側を指定します。この場合、パディングトークンは各シーケンスの左側に追加されます。
- `eos_token` はシーケンスの終わりを表す特殊トークンです。このトークンを `pad_token` に割り当てることで、トークン化中に追加されるパディングトークンもシーケンス終了トークンとみなされます。これはモデルを通じてテキストを生成する際に役立ちます。モデルはパディングトークンに遭遇した後、テキスト生成を停止するタイミングを知ることができるからです。
- `tokenizer.add_special_tokens...` はトークナイザーの語彙に3つの特殊トークンを追加します。これらのトークンはトークナイザーを使用するアプリケーションで特定の目的を果たす可能性があります。例えば、入力の終わり、指示、または対話システムでの応答をマークするために使用できます。

関数が実行された後、`tokenizer` オブジェクトが初期化され、テキストのトークン化に使用する準備が整います。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("databricks/dolly-v2-3b",
                                          padding_side="left")

tokenizer.pad_token = tokenizer.eos_token
tokenizer.add_special_tokens({"additional_special_tokens":
                              [END_KEY, INSTRUCTION_KEY, RESPONSE_KEY_NL]})

### [TASK2-1] Hugging Face のモデルハブから事前学習済みモデルを読み込むコードを以下に挿入してください。

In [ ]:
#############
## TASK 2-1 START ##
#############

print('TASK2-1: START')

## [TASK2-1] Insert code to load a pre-trained model from Hugging Face's model hub. ##
model = AutoModelForCausalLM.from_pretrained(
    "databricks/dolly-v2-3b",
    # use_cache=False,
    device_map="auto", #"balanced",
    load_in_8bit=True,
)

print('TASK2-1: FINISH')

#############
## TASK 2-1 FINISH ##
#############

### <a name="#step3.1">ステップ 3.1: トレーニング用のモデル準備</a>
パラメータ効率の良いファインチューニング（PEFT）を使用して INT8 モデルをトレーニングする前に、いくつかの前処理が必要です。そのため、`prepare_model_for_int8_training` というユーティリティ関数をインポートします。この関数は以下を行います：

- 安定性のために、INT8 でないすべてのモジュールを完全精度（FP32）にキャストします。
- 入力埋め込み層にフォワードフックを追加して、入力隠れ状態の勾配計算を可能にします。
- よりメモリ効率の良いトレーニングのために勾配チェックポイントを有効にします。

In [ ]:
model.resize_token_embeddings(len(tokenizer))

`preprocess_batch` 関数を使用して、バッチのテキストフィールドを前処理し、指定された最大長に基づいてトークン化、切り詰め、その他の関連操作を適用します。このフィールドはデータのバッチ、トークナイザー、および最大長を入力として受け取ります。

詳細については、`mlu_utils/helpers.py` ファイルを参照してください。

In [ ]:
from functools import partial
from utils.helpers import mlu_preprocess_batch

MAX_LENGTH = 256
_preprocessing_function = partial(mlu_preprocess_batch, max_length=MAX_LENGTH, tokenizer=tokenizer)

次に、前処理関数をデータセットの各バッチに適用し、テキストフィールドを適切に修正します。マップ操作はバッチ処理で実行され、instruction、response、および text 列はデータセットから削除されます。最後に、`processed_dataset` は input_ids フィールドの長さに基づいて `sagemaker_faqs_dataset` をフィルタリングして作成され、指定された `MAX_LENGTH` 内に収まるようにします。

In [ ]:
encoded_sagemaker_faqs_dataset = sagemaker_faqs_dataset.map(
        _preprocessing_function,
        batched=True,
        remove_columns=["instruction", "response", "text"],
)

processed_dataset = encoded_sagemaker_faqs_dataset.filter(lambda rec: len(rec["input_ids"]) < MAX_LENGTH)

評価のためにデータセットを `train` と `test` に分割します。

In [ ]:
split_dataset = processed_dataset.train_test_split(test_size=14, seed=0)
split_dataset

## <a name="#step4">ステップ 4: トレーナーの定義と LLM のファインチューニング</a>

モデルを効率的にファインチューニングするために、この実習では [LoRA: Low-Rank Adaptation](https://arxiv.org/abs/2106.09685) を使用します。LoRA は事前学習済みモデルの重みを凍結し、Transformer アーキテクチャの各層にトレーニング可能なランク分解行列を注入することで、下流タスクのためのトレーニング可能なパラメータの数を大幅に削減します。Adam でファインチューニングされた GPT-3 175B と比較して、LoRA はトレーニング可能なパラメータの数を 10,000 倍削減し、GPU メモリ要件を 3 倍削減できます。


### <a name="#step4.1">ステップ 4.1: LoraConfig の定義と LoRA モデルの読み込み</a>

[huggingface 🤗 PEFT: 最先端のパラメータ効率の良いファインチューニング](https://github.com/huggingface/peft) から LoRA クラス `LoraConfig` を使用します。`LoraConfig` 内で、以下のパラメータを指定します：

- `r`、低ランク行列の次元
- `lora_alpha`、低ランク行列のスケーリング係数
- `lora_dropout`、LoRA 層のドロップアウト確率

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_int8_training, TaskType

MICRO_BATCH_SIZE = 8
BATCH_SIZE = 64
GRADIENT_ACCUMULATION_STEPS = BATCH_SIZE // MICRO_BATCH_SIZE
LORA_R = 256 # 512
LORA_ALPHA = 512 # 1024
LORA_DROPOUT = 0.05

# Define LoRA Config
lora_config = LoraConfig(
                 r=LORA_R,
                 lora_alpha=LORA_ALPHA,
                 lora_dropout=LORA_DROPOUT,
                 bias="none",
                 task_type="CAUSAL_LM"
)

`get_peft_model` 関数を使用して、提供された `lora_config` 設定に基づいて LoRA フレームワークでモデルを初期化します。これにより、モデルは LoRA 最適化アプローチの利点と機能を取り入れることができます。

In [ ]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

示されているように、LoRA のみのトレーニング可能なパラメータは全重みの約 3 パーセントであり、はるかに効率的です。

### <a name="#step4.2">ステップ 4.2: データコレーターの定義</a>

DataCollator は、データセットからサンプルのリストを取得し、それらを PyTorch テンソルの辞書としてバッチにまとめる huggingface🤗 transformers 関数です。

`DataCollatorForCompletionOnlyLM` を使用します。これは Transformers ライブラリのベース `DataCollatorForLanguageModeling` クラスの機能を拡張したものです。このカスタムコレーターは、入力テキストでプロンプトの後に応答が続き、ラベルがそれに応じて修正される例を処理するように設計されています。

実装については、`utils/helpers.py` を参照してください。

In [ ]:
from utils.helpers import MLUDataCollatorForCompletionOnlyLM

data_collator = MLUDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer, mlm=False, return_tensors="pt", pad_to_multiple_of=8
)

### <a name="#step5.3">ステップ 5.3: トレーナーの定義</a>

LLM をファインチューニングするには、トレーナーを定義する必要があります。まず、トレーニング引数を定義します。

In [ ]:
EPOCHS = 10
LEARNING_RATE = 1e-4
MODEL_SAVE_FOLDER_NAME = "dolly-3b-lora"

training_args = TrainingArguments(
                    output_dir=MODEL_SAVE_FOLDER_NAME,
                    fp16=True,
                    per_device_train_batch_size=1,
                    per_device_eval_batch_size=1,
                    learning_rate=LEARNING_RATE,
                    num_train_epochs=EPOCHS,
                    logging_strategy="steps",
                    logging_steps=100,
                    evaluation_strategy="steps",
                    eval_steps=100,
                    save_strategy="steps",
                    save_steps=20000,
                    save_total_limit=10,
)

ここが魔法が起こる場所です！定義されたモデル、トークナイザー、トレーニング引数、データコレーター、およびトレーニング/評価データセットでトレーナーを初期化します。

In [ ]:
trainer = Trainer(
        model=model,
        tokenizer=tokenizer,
        args=training_args,
        train_dataset=split_dataset['train'],
        eval_dataset=split_dataset["test"],
        data_collator=data_collator,
)
model.config.use_cache = False  # silence the warnings. Please re-enable for inference!

## [TASK2-2] Trainer オブジェクトをファインチューニングするコードを以下に挿入してください。

**注意: トレーニングには約15分かかります。**

In [ ]:
#############
## TASK 2-2 START ##
#############

print('TASK2-2: START')

## [TASK2-2] Insert code to fine-tune for Trainer object. please check huggingface official doc. ##
trainer.train()

print('TASK2-2: FINISH')

#############
## TASK 2-2 FINISH ##
#############

### <a name="#step4.3">ステップ 4.3: ファインチューニングされたモデルの保存</a>


トレーニングが完了したら、[`transformers.PreTrainedModel.save_pretrained`] 関数を使用してモデルをディレクトリに保存できます。
この関数は、トレーニングされた増分 🤗 PEFT 重み（adapter_model.bin）のみを保存するため、モデルの保存、転送、読み込みが非常に効率的です。

In [ ]:
trainer.model.save_pretrained(MODEL_SAVE_FOLDER_NAME)

ファインチューニングしたばかりの完全なモデルを保存したい場合は、[`transformers.trainer.save_model`] 関数を使用できます。同時に、トレーニング引数をトレーニング済みモデルと一緒に保存します。

In [ ]:
trainer.save_model()

In [ ]:
trainer.model.config.save_pretrained(MODEL_SAVE_FOLDER_NAME)

トレーニング済みモデルと一緒にトークナイザーを保存します。

In [ ]:
tokenizer.save_pretrained(MODEL_SAVE_FOLDER_NAME)

## <a name="#step5">ステップ 5: ファインチューニングされたモデルのデプロイ</a>

### <a name="step5">デプロイパラメータの概要</a>

Deep Java Library（DJL）を使用して Amazon SageMaker Python SDK でデプロイするには、以下のパラメータで `Model` クラスをインスタンス化する必要があります：
```{python}
model = Model(
    image_uri,
    model_data=...,
    predictor_cls=...,
    role=aws_role
)
```
- `image_uri`: 使用する深層学習フレームワークとバージョンを表す Docker イメージ URI。
- `model_data`: Amazon Simple Storage Service（Amazon S3）バケット内のファインチューニングされた LLM モデルアーティファクトの場所。モデルのパラメータ、アーキテクチャ、および必要なアーティファクトを含む TAR GZ ファイルへのパスを指定します。
- `predictor_cls`: これは単なる JSON 入出力の予測子であり、DJL 関連のものではありません。詳細については、[sagemaker.djl_inference.DJLPredictor](https://sagemaker.readthedocs.io/en/stable/frameworks/djl/sagemaker.djl_inference.html#djlpredictor) を参照してください。
- `role`: モデルデータを含む S3 バケットなどのリソースへのアクセスに必要な権限を提供する AWS Identity and Access Management（IAM）ロール ARN。

### <a name="step5.1">ステップ 5.1: SageMaker パラメータのインスタンス化</a>

Amazon SageMaker セッションを初期化し、SageMaker ロールや AWS リージョンなどの AWS 環境に関連する情報を取得します。また、SageMaker セッションのリージョンを使用して、「djl-deepspeed」フレームワークの特定バージョンのイメージ URI を指定します。イメージ URI は、Amazon SageMaker や Amazon Elastic Container Registry（Amazon ECR）などの様々な AWS サービスで使用できる特定の Docker コンテナイメージの一意の識別子です。

In [ ]:
# installing sagemaker library
!pip3 install sagemaker==2.237.1

注意: 警告が表示されるかもしれませんが、そのまま続けてください。

In [ ]:
import boto3
import json
import sagemaker.djl_inference
from sagemaker.session import Session
from sagemaker import image_uris
from sagemaker import Model

sagemaker_session = Session()
print("sagemaker_session: ", sagemaker_session)

aws_role = sagemaker_session.get_caller_identity_arn()
print("aws_role: ", aws_role)

aws_region = boto3.Session().region_name
print("aws_region: ", aws_region)

image_uri = image_uris.retrieve(framework="djl-deepspeed",
                                version="0.22.1",
                                region=sagemaker_session._region_name)
print("image_uri: ", image_uri)


### <a name="step5.2">ステップ 5.2: モデルアーティファクトの作成</a> ###

モデルアーティファクトを S3 バケットにアップロードするには、モデルのパラメータを含む TAR GZ ファイルを作成する必要があります。まず、`lora_model` という名前のディレクトリと `dolly-3b-lora` という名前のサブディレクトリを作成します。"-p" オプションは、中間ディレクトリが存在しない場合にコマンドがそれらを作成することを保証します。次に、LoRA チェックポイント `adapter_model.bin` と `adapter_config.json` を `dolly-3b-lora` にコピーします。ベースの Dolly モデルは実行時に Hugging Face Hub からダウンロードされます。

In [ ]:
%%bash
rm -rf lora_model
mkdir -p lora_model
mkdir -p lora_model/dolly-3b-lora
cp dolly-3b-lora/adapter_config.json lora_model/dolly-3b-lora/
cp dolly-3b-lora/adapter_model.bin lora_model/dolly-3b-lora/

次に、`serving.properties` で [DJL サービング構成オプション](https://docs.aws.amazon.com/sagemaker/latest/dg/large-model-inference-configuration.html) を設定します。Jupyter の `%%writefile` マジックコマンドを使用して、以下の内容を lora_model/serving.properties という名前のファイルに書き込むことができます。
- `engine=Python`: この行は、サービングに使用するエンジンを指定します。
- `option.entryPoint=model.py`: この行は、サービングプロセスのエントリポイントを指定し、model.py に設定されています。
- `option.adapter_checkpoint=dolly-3b-lora`: この行は、アダプターのチェックポイントを dolly-3b-lora に設定します。チェックポイントは通常、モデルまたはそのパラメータの保存された状態を表します。
- `option.adapter_name=dolly-lora`: この行は、アダプターの名前を dolly-lora に設定します。これはモデルとサービングインフラストラクチャ間のインターフェースを支援するコンポーネントです。

In [ ]:
%%writefile lora_model/serving.properties
engine=Python
option.entryPoint=model.py
option.adapter_checkpoint=dolly-3b-lora
option.adapter_name=dolly-lora

モデルアーティファクトには環境要件ファイルも必要です。`lora_model/requirements.txt` という名前のファイルを作成し、通常 `pip` などのパッケージマネージャーで使用される Python パッケージ要件のリストを記述します。

In [ ]:
%%writefile lora_model/requirements.txt
accelerate>=0.16.0,<1
bitsandbytes==0.39.0
click>=8.0.4,<9
datasets>=2.10.0,<3
deepspeed>=0.8.3,<0.9
faiss-cpu==1.7.4
ipykernel==6.22.0
scipy==1.11.1
torch>=2.0.0
transformers==4.28.1
peft==0.3.0
pytest==7.3.2

### <a name="step5.3">ステップ 5.3: 推論スクリプトの作成</a>

ファインチューニングノートブックと同様に、カスタムパイプライン `InstructionTextGenerationPipeline` が定義されています。コードは `utils/deployment_model.py` で提供されています。

これらの推論関数を `lora_model/model.py` に保存します。

In [ ]:
%%bash
cp utils/deployment_model.py lora_model/model.py

### <a name="step5.4">ステップ 5.4: モデルアーティファクトを Amazon S3 にアップロード</a>

lora_model ディレクトリの圧縮された tarball アーカイブを作成し、lora_model.tar.gz として保存します。

In [ ]:
%%bash
tar -cvzf lora_model.tar.gz lora_model/

lora_model.tar.gz ファイルを指定された S3 バケットにアップロードします。

In [ ]:
import boto3
import json
import sagemaker.djl_inference
from sagemaker.session import Session
from sagemaker import image_uris
from sagemaker import Model

s3 = boto3.resource('s3')
s3_client = boto3.client('s3')

# Get the name of the bucket with prefix lab-code
for bucket in s3.buckets.all():
    if bucket.name.startswith('artifact'):
        mybucket = bucket.name
        print(mybucket)

response = s3_client.upload_file("lora_model.tar.gz", mybucket, "lora_model.tar.gz")

### <a name="step5.5">ステップ 5.5: モデルのデプロイ</a> ###

いよいよ SageMaker Python SDK を使用してファインチューニングされた LLM をデプロイする時が来ました。SageMaker Python SDK の `Model` クラスは以下のパラメータでインスタンス化されます：

- `image_uri`: 使用する深層学習フレームワークとバージョンを表す Docker イメージ URI。
- `model_data`: S3 バケット内のファインチューニングされた LLM モデルアーティファクトの場所。モデルのパラメータ、アーキテクチャ、および必要なアーティファクトを含む TAR GZ ファイルへのパスを指定します。
- `predictor_cls`: これは単なる JSON 入出力の予測子であり、DJL 関連のものではありません。詳細については、[sagemaker.djl_inference.DJLPredictor](https://sagemaker.readthedocs.io/en/stable/frameworks/djl/sagemaker.djl_inference.html#djlpredictor) を参照してください。
- `role`: モデルデータを含む S3 バケットなどのリソースへのアクセスに必要な権限を提供する IAM ロール ARN。

In [ ]:
model_data="s3://{}/lora_model.tar.gz".format(mybucket)

## [TASK2-3] Model オブジェクトを定義するコードを以下に挿入してください。

In [ ]:
#############
## TASK 2-3 START ##
#############

print('TASK2-3: START')

## [TASK2-3] Insert code here to define Model object. ##
model = Model(image_uri=image_uri,
              model_data=model_data,
              predictor_cls=sagemaker.djl_inference.DJLPredictor,
              role=aws_role)

print('TASK2-3: FINISH')

#############
## TASK 2-3 FINISH ##
#############

## [TASK2-4] モデルをデプロイするコードを以下に挿入してください。
**デプロイオプションは initial_instance_count=1, instance_type="ml.g4dn.2xlarge" にする必要があります**

In [ ]:
#############
## TASK 2-4 START ##
#############

print('TASK2-4: START')

%time
## [TASK2-4] Insert code here to deploy model,deploy option should be initial_instance_count=1, instance_type="ml.g4dn.2xlarge" ##
predictor = model.deploy(1, "ml.g4dn.2xlarge")

print('TASK2-4: FINISH')

#############
## TASK 2-4 FINISH ##
#############

**注意: デプロイには 10 分くらい時間がかかります**

### <a name="step6">ステップ 6: デプロイされた推論のテスト</a>

[predictor.predict](https://sagemaker.readthedocs.io/en/stable/api/inference/predictors.html#sagemaker.predictor.Predictor.predict) で推論エンドポイントをテストします。

In [ ]:
outputs = predictor.predict({"inputs": "What solutions come pre-built with Amazon SageMaker JumpStart?"})

In [ ]:
from IPython.display import Markdown
Markdown(outputs)